## Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Ma'ta/Disease and symptoms dataset (2).zip"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(246945, 378)


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


## Clean data

In [ ]:
y = df["diseases"]
X = df.drop(columns=["diseases"])

dead_cols = X.columns[X.sum() == 0]
print("dead columns:", len(dead_cols))

X = X.drop(columns=list(dead_cols) + ["regurgitation"])
print(X.shape)

dead columns: 49
(246945, 327)


In [ ]:
data = X.copy()
data["diseases"] = y

unique = data.groupby(list(data.columns), as_index=False).size()
unique = unique.rename(columns={"size": "weight"})

print("rows before:", len(data))
print("unique rows:", len(unique))
print("max weight:", unique["weight"].max())

/tmp/ipykernel_581/4089436945.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  unique = data.groupby(list(data.columns), as_index=False).size()
/tmp/ipykernel_581/4089436945.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  unique = data.groupby(list(data.columns), as_index=False).size()
/tmp/ipykernel_581/4089436945.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis

rows before: 246945
unique rows: 189647
max weight: 20


In [ ]:
counts = unique["diseases"].value_counts()

for n in [1, 2, 5, 10]:
    print(f"diseases with < {n} unique rows:", (counts < n).sum())
print("total diseases:", len(counts))

diseases with < 1 unique rows: 0
diseases with < 2 unique rows: 46
diseases with < 5 unique rows: 115
diseases with < 10 unique rows: 186
total diseases: 773


In [ ]:
keep = counts[counts >= 5].index
unique = unique[unique["diseases"].isin(keep)]

print("diseases left:", unique["diseases"].nunique())
print("rows left:", len(unique))

diseases left: 658
rows left: 189409


## Split

In [ ]:
from sklearn.model_selection import train_test_split
train, temp = train_test_split(
    unique, test_size=0.40, stratify=unique["diseases"], random_state=42
)
val, test = train_test_split(
    temp, test_size=0.50, stratify=temp["diseases"], random_state=42
)

print("train:", len(train), "| val:", len(val), "| test:", len(test))
print("diseases in val:", val["diseases"].nunique(), "| in test:", test["diseases"].nunique())

train: 113645 | val: 37882 | test: 37882
diseases in val: 658 | in test: 658


## Masking

In [ ]:
import numpy as np

symptom_cols = [c for c in unique.columns if c not in ("diseases", "weight")]
rng = np.random.default_rng(42)

def keep_k(part, k):
    M = part[symptom_cols].to_numpy(dtype=np.int8)
    k = np.minimum(k, M.sum(axis=1))
    r = rng.random(M.shape, dtype=np.float32)
    r[M == 0] = 2.0
    cut = np.sort(r, axis=1)[np.arange(len(M)), k - 1]
    return ((r <= cut[:, None]) & (M == 1)).astype(np.int8)

In [ ]:
copies = []
for _ in range(3):
    n = train[symptom_cols].sum(axis=1).to_numpy()
    k = rng.integers(1, n + 1)
    copies.append(keep_k(train, k))

X_train = np.vstack([train[symptom_cols].to_numpy(dtype=np.int8)] + copies)
y_train = np.concatenate([train["diseases"].to_numpy()] * 4)
w_train = np.concatenate([train["weight"].to_numpy()] * 4)

print(X_train.shape, "| avg symptoms per row:", X_train.sum(axis=1).mean().round(2))

(454580, 327) | avg symptoms per row: 3.75


In [ ]:
y_val = val["diseases"].to_numpy()
val_sets = {k: keep_k(val, k) for k in [1, 2, 3, 4]}

for k, Xv in val_sets.items():
    print(k, "symptoms ->", Xv.shape)

1 symptoms -> (37882, 327)
2 symptoms -> (37882, 327)
3 symptoms -> (37882, 327)
4 symptoms -> (37882, 327)


## Model: Bernoulli naive Bayes, rare diseases oversampled to 30 rows

In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import top_k_accuracy_score, precision_recall_fscore_support

def ece(p, y, classes):
    conf = p.max(axis=1)
    ok = classes[p.argmax(axis=1)] == y
    bins = np.minimum((conf * 10).astype(int), 9)
    return sum(abs(conf[bins == b].mean() - ok[bins == b].mean()) * (bins == b).mean()
               for b in range(10) if (bins == b).any())

def evaluate(model, name, sets, y_true):
    for k, Xs in sets.items():
        p = model.predict_proba(Xs)
        m1, m3, m5 = [top_k_accuracy_score(y_true, p, k=t, labels=model.classes_) for t in (1, 3, 5)]
        pred = model.classes_[p.argmax(axis=1)]
        pr, rc, f1, _ = precision_recall_fscore_support(y_true, pred, average="macro", zero_division=0)
        e = ece(p, y_true, model.classes_)
        print(f"{name} | {k} symptoms | M1 {m1:.3f} | M3 {m3:.3f} | M5 {m5:.3f} | P {pr:.3f} | R {rc:.3f} | F1 {f1:.3f} | ECE {e:.3f}")

In [ ]:
cnt = train["diseases"].value_counts()
small = cnt[cnt < 30]
rng_os = np.random.default_rng(0)
idx = [np.arange(len(y_train))]
for d, c in small.items():
    rows = np.where(y_train == d)[0]
    idx.append(rng_os.choice(rows, 4 * (30 - c)))
idx = np.concatenate(idx)
model = BernoulliNB(alpha=1.0)
model.fit(X_train[idx], y_train[idx], sample_weight=w_train[idx])
evaluate(model, "NB", val_sets, y_val)

NB | 1 symptoms | M1 0.083 | M3 0.234 | M5 0.360 | P 0.088 | R 0.066 | F1 0.045 | ECE 0.029
NB | 2 symptoms | M1 0.343 | M3 0.623 | M5 0.755 | P 0.442 | R 0.388 | F1 0.338 | ECE 0.042
NB | 3 symptoms | M1 0.577 | M3 0.831 | M5 0.910 | P 0.617 | R 0.628 | F1 0.580 | ECE 0.037
NB | 4 symptoms | M1 0.725 | M3 0.912 | M5 0.959 | P 0.741 | R 0.742 | F1 0.716 | ECE 0.025


In [ ]:
def rare_m3(model, Xs, y_true):
    p = model.predict_proba(Xs)
    top3 = model.classes_[np.argsort(-p, axis=1)[:, :3]]
    hit = (top3 == y_true[:, None]).any(axis=1)
    rare = cnt.reindex(y_true).to_numpy() <= 10
    balanced = pd.Series(hit).groupby(y_true).mean().mean()
    return hit[rare].mean(), balanced

r, b = rare_m3(model, val_sets[3], y_val)
print(f"rare M3 {r:.3f} | balanced M3 {b:.3f}")

rare M3 0.865 | balanced M3 0.830


## Overfitting check

In [ ]:
y_tr = train["diseases"].to_numpy()
for k in [2, 3, 4]:
    tr = top_k_accuracy_score(y_tr, model.predict_proba(keep_k(train, k)), k=3, labels=model.classes_)
    va = top_k_accuracy_score(y_val, model.predict_proba(val_sets[k]), k=3, labels=model.classes_)
    print(f"{k} symptoms | train M3 {tr:.3f} | val M3 {va:.3f} | gap {tr - va:.3f}")

2 symptoms | train M3 0.633 | val M3 0.623 | gap 0.010
3 symptoms | train M3 0.835 | val M3 0.831 | gap 0.005
4 symptoms | train M3 0.915 | val M3 0.912 | gap 0.003


## Test set (run once)

In [ ]:
y_test = test["diseases"].to_numpy()
test_sets = {k: keep_k(test, k) for k in [1, 2, 3, 4]}
test_sets["all"] = test[symptom_cols].to_numpy(dtype=np.int8)

evaluate(model, "NB test", test_sets, y_test)
r, b = rare_m3(model, test_sets[3], y_test)
print(f"rare M3 {r:.3f} | balanced M3 {b:.3f}")

NB test | 1 symptoms | M1 0.086 | M3 0.234 | M5 0.361 | P 0.091 | R 0.062 | F1 0.045 | ECE 0.030
NB test | 2 symptoms | M1 0.340 | M3 0.625 | M5 0.756 | P 0.432 | R 0.379 | F1 0.327 | ECE 0.045
NB test | 3 symptoms | M1 0.582 | M3 0.831 | M5 0.911 | P 0.623 | R 0.630 | F1 0.584 | ECE 0.034
NB test | 4 symptoms | M1 0.726 | M3 0.913 | M5 0.960 | P 0.739 | R 0.739 | F1 0.714 | ECE 0.023
NB test | all symptoms | M1 0.843 | M3 0.953 | M5 0.977 | P 0.809 | R 0.801 | F1 0.785 | ECE 0.007
rare M3 0.865 | balanced M3 0.832


## Save

In [ ]:
import joblib

OUT = "/content/drive/MyDrive/Ma'ta/"
joblib.dump({"model": model, "columns": symptom_cols}, OUT + "classifier.joblib")
print("saved")

saved
